# Module 01: Data Engineering for MLOps

**What you'll learn:**
- Why data quality is the foundation of any ML system
- How to generate realistic synthetic data
- How to validate data automatically
- How to engineer features for time-series forecasting
- How to version and store features for reproducibility

**Time:** ~1.5 hours

## 1. Why Data Engineering Matters

> "Your model is only as good as your data."

In a Jupyter notebook, data problems are annoying. In production, they're **catastrophic**. Here's why:

- **Silent failures**: A sensor goes offline and starts sending zeros. Your model doesn't crash — it just gives terrible predictions. Nobody notices for weeks.
- **Data drift**: Seasons change, building occupancy shifts, new equipment is installed. The data your model trained on no longer represents reality.
- **Schema changes**: An upstream system renames a column. Your pipeline breaks at 3 AM.

Data engineering gives you **three lines of defense**:
1. **Data Validation** — Catch problems before they reach your model
2. **Feature Engineering** — Transform raw data into useful model inputs
3. **Feature Store** — Version and track features for reproducibility

## 2. Our Energy Dataset

We're working with **hourly energy demand data** for buildings. Each row represents one hour of electricity consumption for one building.

Let's generate some synthetic data and explore it:

In [ ]:
import sys
sys.path.insert(0, '../src')

from energy_forecast.data.synthetic import SyntheticDataGenerator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Generate realistic energy demand data
gen = SyntheticDataGenerator(
    num_buildings=5,
    start_date="2022-01-01",
    end_date="2023-12-31",
    random_seed=42
)
df = gen.generate()

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nDate range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"\nBuildings: {df['building_id'].nunique()}")
df.head(10)

### Exploring Patterns

Energy demand has strong **temporal patterns**. Let's visualize them:

In [ ]:
# Pick one building to study
building = df[df['building_id'] == df['building_id'].unique()[0]].copy()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Daily pattern (1 week)
week = building.head(168)  # 7 days * 24 hours
axes[0].plot(week['timestamp'], week['energy_demand_kwh'], color='steelblue')
axes[0].set_title('Daily Pattern (One Week)', fontsize=14)
axes[0].set_ylabel('Energy Demand (kWh)')
axes[0].grid(True, alpha=0.3)

# Monthly pattern
month = building.head(720)  # ~30 days
axes[1].plot(month['timestamp'], month['energy_demand_kwh'], color='coral')
axes[1].set_title('Weekly Pattern (One Month)', fontsize=14)
axes[1].set_ylabel('Energy Demand (kWh)')
axes[1].grid(True, alpha=0.3)

# Full period
axes[2].plot(building['timestamp'], building['energy_demand_kwh'], alpha=0.6, color='green')
axes[2].set_title('Seasonal Pattern (Full Period)', fontsize=14)
axes[2].set_ylabel('Energy Demand (kWh)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Notice: peaks in morning/evening (daily), lower on weekends (weekly), higher in winter/summer (seasonal)")

## 3. Data Validation — Catching Problems Early

### What can go wrong?

In production, data breaks in predictable ways:
- **Missing values**: Sensors go offline
- **Out-of-range values**: A temperature sensor reads 999°C
- **Negative values**: Energy demand can't be negative
- **Duplicates**: The same reading sent twice
- **Schema changes**: Expected columns are missing

**Data validation** runs automated checks on every batch of data BEFORE it reaches your model. Think of it as a bouncer at the door — bad data doesn't get in.

Let's see it in action:

In [ ]:
from energy_forecast.data.validator import DataValidator

validator = DataValidator()

# Validate our clean data
result = validator.validate_raw_data(df)
print("=== Validation on CLEAN data ===")
print(f"Is valid: {result.is_valid}")
print(f"Errors: {result.errors}")
print(f"Warnings: {result.warnings}")
print(f"Stats: {result.stats}")

print("\n" + "=" * 50)

# Now let's BREAK the data and see if validation catches it
bad_df = df.copy()
bad_df.loc[0:10, 'energy_demand_kwh'] = -5       # Negative energy!
bad_df.loc[100:110, 'temperature'] = 999          # Impossible temperature
bad_df.loc[200:210, 'energy_demand_kwh'] = None    # Missing values

result = validator.validate_raw_data(bad_df)
print("\n=== Validation on CORRUPTED data ===")
print(f"Is valid: {result.is_valid}")
for error in result.errors:
    print(f"  ERROR: {error}")

The validator caught all three problems automatically. In production, this would prevent corrupted data from reaching your model.

## 4. Feature Engineering — Teaching Your Model to See Patterns

Raw data (timestamp, temperature, energy demand) isn't enough for a model. We need to create **features** — new columns that encode the patterns we saw in the plots.

### Why each feature type matters:

| Feature Type | What It Captures | Example |
|-------------|-----------------|--------|
| **Lag features** | "What was demand N hours ago?" | `demand_lag_24` = yesterday same hour |
| **Rolling stats** | "What's the recent trend?" | `demand_rolling_mean_6h` |
| **Cyclical time** | "Where are we in the day/week/year?" | `hour_sin`, `hour_cos` |
| **Calendar** | "Is it a special day?" | `is_weekend`, `is_holiday` |
| **Weather** | "How does weather affect demand?" | `heating_degree_days` |

### Why sin/cos for time?

If we just use `hour = 0, 1, 2, ... 23`, the model thinks hour 23 and hour 0 are far apart (distance = 23). But they're actually neighbors! Sin/cos encoding puts them close together on a circle.

In [ ]:
from energy_forecast.features.engineering import FeatureEngineer

engineer = FeatureEngineer()

# Step 1: Time features (cyclical encoding)
df_feat = engineer.create_time_features(df.copy())
time_cols = [c for c in df_feat.columns if 'sin' in c or 'cos' in c]
print(f"Time features added: {time_cols}")

# Step 2: Lag features
df_feat = engineer.create_lag_features(df_feat, 'energy_demand_kwh', [1, 2, 3, 6, 12, 24, 168])
lag_cols = [c for c in df_feat.columns if 'lag' in c]
print(f"\nLag features added: {lag_cols}")

# Step 3: Rolling statistics
df_feat = engineer.create_rolling_features(df_feat, 'energy_demand_kwh', [6, 12, 24])
roll_cols = [c for c in df_feat.columns if 'rolling' in c]
print(f"\nRolling features added ({len(roll_cols)}): {roll_cols[:6]}...")

# Step 4: Weather features
df_feat = engineer.create_weather_features(df_feat)
weather_cols = [c for c in df_feat.columns if 'degree' in c or 'interaction' in c]
print(f"\nWeather features added: {weather_cols}")

# Step 5: Calendar features
df_feat = engineer.create_calendar_features(df_feat)
cal_cols = [c for c in df_feat.columns if 'is_' in c or 'season' in c]
print(f"\nCalendar features added: {cal_cols}")

# Drop NaN rows (from lag/rolling features)
df_feat = df_feat.dropna()
all_features = engineer.get_feature_names(df_feat)
print(f"\n=== Total features: {len(all_features)} ===")

## 5. Feature Store — Versioning Your Features

Why version features?
- **Reproducibility**: "Which features did I use to train model v3?"
- **Rollback**: If new features break the model, go back to the working version
- **Collaboration**: Team members use the same feature definitions

In [ ]:
from energy_forecast.features.store import FeatureStore
import tempfile

store = FeatureStore(base_path=tempfile.mkdtemp())

# Save versioned features
store.save_features(df_feat, name="energy_features", version="v1",
                    metadata={"buildings": 5, "date_range": "2022-2023", "n_features": len(all_features)})

# Load them back
loaded = store.load_features("energy_features", version="v1")
print(f"Saved shape: {df_feat.shape}")
print(f"Loaded shape: {loaded.shape}")
print(f"Shapes match: {df_feat.shape == loaded.shape}")
print(f"\nMetadata: {store.get_metadata('energy_features', 'v1')}")
print(f"Versions available: {store.list_versions('energy_features')}")

## 6. Data Splitting — Why Time-Based Splits Matter

**Critical rule for time series**: Never use random train/test splits!

Random splits let the model see future data during training. This is called **data leakage** — your metrics look great in the notebook but the model fails in production.

**Correct approach**: Split by time. Everything before a date = training. Everything after = testing.

```
Time →  [=======TRAIN=======][===VAL===][===TEST===]
        Jan 2022             Oct 2022    Apr 2023
```

In [ ]:
from energy_forecast.data.processor import DataProcessor

processor = DataProcessor(config={
    'lag_features': [1, 2, 3, 24],
    'rolling_windows': [6, 24],
    'rolling_stats': ['mean'],
    'drop_na': True
})

features_df = processor.create_features(df.copy())
split = processor.split_data(features_df)

print(f"Train: {split.X_train.shape} ({split.X_train.shape[0] / len(features_df) * 100:.0f}%)")
print(f"Val:   {split.X_val.shape} ({split.X_val.shape[0] / len(features_df) * 100:.0f}%)")
print(f"Test:  {split.X_test.shape} ({split.X_test.shape[0] / len(features_df) * 100:.0f}%)")
print("\nNo future leakage — all splits are sequential in time!")

## 7. Exercises

1. **Add a new feature**: Create a `demand_change_rate` feature that measures the hourly change in demand (current - previous hour). Add it to the FeatureEngineer.

2. **Custom validation**: Write a validation rule that flags buildings with more than 5% missing hourly readings.

3. **Correlation analysis**: Plot the correlation matrix between all features and `energy_demand_kwh`. Which features are most predictive?

## 8. Key Takeaways

- **Data validation** is your first line of defense — catch problems before they reach your model
- **Feature engineering** encodes domain knowledge (time patterns, weather effects) into model inputs
- **Time-based splits** prevent data leakage in time series
- **Feature stores** give you versioning and reproducibility
- Every step in the data pipeline should be automated and reproducible

**Next: [Notebook 02 - Experiment Tracking with MLflow](./02_experiment_tracking.ipynb)**